In [2]:
import os
import sys
import json
import pickle
import warnings
import numpy as np
import sympy as sp
import pandas as pd
import proplot as pplt
from IPython.display import display, HTML
sys.path.insert(0,'..')
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
MODELSDIR = CONFIGS['filepaths']['models']
SRCONFIG  = CONFIGS['experiments']['sr']
SEEDS     = SRCONFIG['seeds']

In [4]:
VARDICT = {
    'rh':       r'\widehat{\mathrm{RH}}',
    'thetae':   r'\widehat{\theta}_{e}',
    'thetaestar':r'{\widehat{\theta}_{e}^{*}}',
    'bl':       r'\mathrm{B_L}',
    'lf':       r'\mathrm{LF}',
    'shf':      r'\mathrm{SHF}',
    'lhf':      r'\mathrm{LHF}'}

SYMBOLS  = {k: sp.Symbol(k) for k in VARDICT}

FUNCDICT = {
    'cube':   lambda x: x**3,
    'square': lambda x: x**2,
    'sqrt':   sp.sqrt,
    'abs':    sp.Abs,
    'neg':    lambda x: -x,
    'exp':    sp.exp,
    'log':    sp.log,
    'sin':    sp.sin,
    'cos':    sp.cos,
    'max':    sp.Max,
    'min':    sp.Min}

TERMORDER = {'bl':0,'rh':1,'thetae':2,'thetaestar':3,'lf':4,'shf':5,'lhf':6}

def _to_sympy_expr(eq):
    return sp.sympify(eq, locals={**SYMBOLS, **FUNCDICT})

def _round_numbers(expr, ndigits=4):
    return expr.xreplace({n: sp.Float(round(float(n), ndigits), ndigits) for n in expr.atoms(sp.Float)})

def _term_key(term):
    symbols = term.free_symbols
    if not symbols:
        return (99, str(term))
    names = sorted(s.name for s in symbols)
    return (min(TERMORDER.get(n, 50) for n in names), str(term))

def _ordered_add_terms(expr):
    if isinstance(expr, sp.Add):
        terms = sp.Add.make_args(expr)
        return sp.Add(*sorted(terms, key=_term_key), evaluate=False)
    return expr

def _order_expr(expr):
    if expr.args:
        expr = expr.func(*[_order_expr(arg) for arg in expr.args], evaluate=False)
    if isinstance(expr, sp.Add):
        expr = _ordered_add_terms(expr)
    return expr

def _latex_expr(expr):
    symbolnames = {SYMBOLS[k]: v for k, v in VARDICT.items()}
    latex = sp.latex(expr, symbol_names=symbolnames, mul_symbol='dot')
    latex = latex.replace(r'\left', '').replace(r'\right', '')
    latex = ' '.join(latex.split())
    return latex

def prettify(eq):
    try:
        expr = _to_sympy_expr(str(eq).strip())
        expr = _round_numbers(expr, ndigits=4)
        expr = _order_expr(expr)
        return '$' + _latex_expr(expr) + '$'
    except Exception:
        return str(eq).strip()

def load_equations(runname):
    seedframes = {}
    for seed in SEEDS:
        filepath = os.path.join(MODELSDIR, 'sr', f'{runname}_{seed}_equations.csv')
        if not os.path.exists(filepath):
            continue
        df = pd.read_csv(filepath)
        df['seed'] = seed
        seedframes[seed] = df
    return seedframes

In [ ]:
PARETO_COLORS = ['#5BA7DA', '#F2C85E', '#D42028']

for runname in SRCONFIG['runs']:
    seedframes = load_equations(runname)
    if not seedframes:
        print(f'[{runname}] No equations found — run on cluster')
        continue

    sel = {name: spec for name, spec in SRCONFIG['optimizedeqs'].items()
           if spec['runfrom'] == runname and spec.get('refcomplexity') is not None}
    selcomplexities = sorted({spec['refcomplexity'] for spec in sel.values()})
    seeds_here      = sorted(seedframes.keys())
    allcomplexity   = pd.concat(seedframes.values())['complexity']

    fig, ax = pplt.subplots(refwidth=5, refheight=2)
    ax.format(title=f'Pareto front — {runname}', xlabel='Complexity',
              xlim=(0, allcomplexity.max() + 1), ylabel='Loss')
    for i, seed in enumerate(seeds_here):
        df    = seedframes[seed].sort_values('complexity')
        color = PARETO_COLORS[i % len(PARETO_COLORS)]
        ax.plot(df['complexity'], df['loss'], color=color, alpha=0.5, linewidth=1,
                linestyle='--', zorder=1, label='')
        ax.scatter(df['complexity'], df['loss'], color=color, marker='.', zorder=3,
                   label=f'Seed {seed}')
        for c in selcomplexities:
            row = df[df['complexity'] == c]
            if not row.empty:
                ax.scatter([row['complexity'].values[0]], [row['loss'].values[0]],
                           color=color, edgecolors='k', marker='*',
                           markersize=100, linewidths=0.5, zorder=5)
    ax.scatter([], [], color='k', marker='*', markersize=100, label='Selected')
    ax.legend(loc='ur', ncols=1)
    pplt.show()

    if not selcomplexities:
        continue
    eq_rows = []
    for c in selcomplexities:
        eq_names = [spec['description'] for name, spec in sel.items()
                    if spec['refcomplexity'] == c]
        row = {'Complexity': c, 'Model': ', '.join(eq_names)}
        for seed in seeds_here:
            df    = seedframes[seed]
            match = df[df['complexity'] == c]
            row[f'Seed {seed}'] = prettify(match.iloc[0]['equation']) if not match.empty else '—'
        eq_rows.append(row)
    display(HTML(pd.DataFrame(eq_rows).to_html(escape=False, index=False)))

In [6]:
def load_registry():
    pklpath = os.path.join(MODELSDIR, 'sr', 'optimized_equations.pkl')
    if not os.path.exists(pklpath):
        return {}
    with open(pklpath, 'rb') as f:
        return pickle.load(f)

def prettify_optimized(opt):
    ns = {**SYMBOLS, **FUNCDICT}
    ns.update({k: sp.Float(v) for k, v in opt['constants'].items()})
    try:
        expr = eval(opt['form'], {'__builtins__': {}}, ns)
        expr = _round_numbers(expr, ndigits=4)
        expr = _order_expr(expr)
        return r'$\hat{P} = ' + _latex_expr(expr) + '$'
    except Exception as e:
        return f"{opt['form']}  [render error: {e}]"

registry = load_registry()
rows = []
for name, eqspec in SRCONFIG['optimizedeqs'].items():
    opt = registry.get(name)
    if opt is None:
        rows.append({'Model': eqspec['description'], 
                     #'Form': eqspec['form'],
                     'Train MSE': '—', 'Valid MSE': '—'})
    else:
        rows.append({'Model':     eqspec['description'],
                     'Equation':  prettify_optimized(opt),
                     'Train MSE': f"{opt['train_loss']:.4f}",
                     'Valid MSE': f"{opt['valid_loss']:.4f}"})
display(HTML(pd.DataFrame(rows).to_html(escape=False,index=False)))

Model,Equation,Train MSE,Valid MSE
SR-BL,$\hat{P} = (\mathrm{B_L} + 0.5576)^{3} - 0.7477$,0.5899,0.6048
SR-LO,$\hat{P} = 1.0 \cdot e^{\widehat{\mathrm{RH}}} - 0.6507$,0.6246,0.6214
SR-MED,"$\hat{P} = 1.885 \cdot \max(\widehat{\mathrm{RH}}, \widehat{\theta}_{e} - 1.111 \cdot {\widehat{\theta}_{e}^{*}} - 3.649)^{3}$",0.7604,0.7418
SR-HI,"$\hat{P} = 0.7884 \cdot (0.2602 \cdot \max(\mathrm{LF}, \mathrm{SHF}) + \max(\widehat{\mathrm{RH}}, \widehat{\theta}_{e} + 0.7834 \cdot {\widehat{\theta}_{e}^{*}} - 3.059))^{3}$",0.9360,0.9032


## Physical-space equations

Each SR model predicts the normalized log-precipitation $\ell = \sigma_{tp}\,\hat{z} + \mu_{tp}$, where $\hat{z}$ is the raw SR output and $P\ [\mathrm{mm}] = \mathrm{expm1}(\max(0,\,\ell))$. Below, the optimized equations are re-expressed entirely in physical units (B_L in m s⁻², RH in %, θ_e and θ_e\* in K), with each free constant replaced by a physically interpretable named parameter. The symbol ℓ denotes log(1 + P [mm]).

In [7]:
statsfile = os.path.join('..', 'data', 'splits', 'stats.json')
if os.path.exists(statsfile):
    with open(statsfile, 'r', encoding='utf-8') as f:
        STATS = json.load(f)
    print('Loaded stats.json')
else:
    STATS = None
    print('[warn] stats.json not found — run on the cluster to populate this table')

Loaded stats.json


In [8]:
def _phys_sr_bl(constants, stats, targetvar='tp'):
    """cube(bl + a) + b  →  physical form in B_L [m s⁻²]"""
    a, b      = constants['a'], constants['b']
    mu_bl, sig_bl = stats['bl_mean'], stats['bl_std']
    sig_tp, mu_tp = stats[f'{targetvar}_std'], stats[f'{targetvar}_mean']
    BL_crit  = mu_bl - a * sig_bl
    alpha_bl = sig_tp / sig_bl**3
    beta_bl  = sig_tp * b + mu_tp
    latex = (
        r'\ell_{\mathrm{BL}} = \underbrace{' + f'{alpha_bl:.3g}' + r'}_{\alpha_{BL}}'
        r'\cdot\bigl(B_L - \underbrace{(' + f'{BL_crit:.4f}' + r')}_{B_L^{\mathrm{crit}}}\bigr)^3'
        r' + \underbrace{' + f'{beta_bl:.3f}' + r'}_{\beta_{BL}}'
    )
    rows = [
        (r'$B_L^{\mathrm{crit}}$', f'{BL_crit:.4f}', 'm s⁻²', 'Critical buoyancy deficit for onset'),
        (r'$\alpha_{BL}$', f'{alpha_bl:.4g}', '(m s⁻²)⁻³ ℓ', 'Cubic precipitation sensitivity'),
        (r'$\beta_{BL}$', f'{beta_bl:.3f}', 'ℓ', 'Intercept in log-precipitation space'),
    ]
    return latex, rows

def _phys_sr_lo(constants, stats, targetvar='tp'):
    """a * exp(rh) + b  →  physical form in RH [%]"""
    a, b      = constants['a'], constants['b']
    mu_rh, sig_rh = stats['rh_mean'], stats['rh_std']
    sig_tp, mu_tp = stats[f'{targetvar}_std'], stats[f'{targetvar}_mean']
    alpha_lo = sig_tp * a * np.exp(-mu_rh / sig_rh)
    beta_lo  = sig_tp * b + mu_tp
    latex = (
        r'\ell_{\mathrm{LO}} = \underbrace{' + f'{alpha_lo:.3g}' + r'}_{\alpha_{LO}}'
        r'\cdot e^{\,\mathrm{RH}\,/\,\underbrace{' + f'{sig_rh:.1f}' + r'}_{\sigma_{RH}}}'
        r' + \underbrace{' + f'{beta_lo:.3f}' + r'}_{\beta_{LO}}'
        r',\quad \mathrm{RH}\ [\%]'
    )
    rows = [
        (r'$\alpha_{LO}$', f'{alpha_lo:.4g}', 'ℓ', 'Exponential amplitude (absorbs centering)'),
        (r'$\sigma_{RH}$', f'{sig_rh:.2f}', '%', 'RH scale — reciprocal is e-folding sensitivity'),
        (r'$\beta_{LO}$', f'{beta_lo:.3f}', 'ℓ', 'Intercept in log-precipitation space'),
    ]
    return latex, rows

def _phys_sr_med(constants, stats, targetvar='tp'):
    """a * cube(max(rh, thetae + b*thetaestar + c))  →  physical form in θe [K]"""
    a, b, c   = constants['a'], constants['b'], constants['c']
    mu_te,  sig_te  = stats['thetae_mean'],     stats['thetae_std']
    mu_tes, sig_tes = stats['thetaestar_mean'], stats['thetaestar_std']
    mu_rh,  sig_rh  = stats['rh_mean'],         stats['rh_std']
    sig_tp,  mu_tp  = stats[f'{targetvar}_std'], stats[f'{targetvar}_mean']
    kappa      = -b * sig_te / sig_tes
    Theta_crit = mu_te + b * (sig_te / sig_tes) * mu_tes - c * sig_te
    alpha_med  = sig_tp * a / sig_te**3
    rh_scale   = sig_te / sig_rh
    latex = (
        r'\ell_{\mathrm{MED}} = \underbrace{' + f'{alpha_med:.3g}' + r'}_{\alpha_{MED}}'
        r'\cdot\max\!\Bigl('
        r'\underbrace{' + f'{rh_scale:.2f}' + r'}_{\sigma_{\theta_e}/\sigma_{RH}}'
        r'(\mathrm{RH} - \bar{\mathrm{RH}}),\;'
        r'\theta_e - \underbrace{' + f'{kappa:.3f}' + r'}_{\kappa}\theta_e^* - '
        r'\underbrace{' + f'{Theta_crit:.1f}' + r'\ \mathrm{K}}_{\Theta_e^{\mathrm{crit}}}'
        r'\Bigr)^3'
    )
    rows = [
        (r'$\alpha_{MED}$', f'{alpha_med:.4g}', 'K⁻³ ℓ', 'Cubic precipitation sensitivity'),
        (r'$\kappa$', f'{kappa:.3f}', '—', 'θe* weighting; κ≈1 → CAPE-like onset criterion'),
        (r'$\Theta_e^{\mathrm{crit}}$', f'{Theta_crit:.1f}', 'K', 'Critical equivalent pot. temp. for onset'),
        (r'$\sigma_{\theta_e}/\sigma_{RH}$', f'{rh_scale:.3f}', 'K %⁻¹', 'Scale factor mapping RH to θe units'),
    ]
    return latex, rows

def _phys_sr_hi(constants, stats, targetvar='tp'):
    """a * cube(max(rh, thetae + b*thetaestar + c) + max(lf,shf)*d)  →  physical form"""
    a, b, c, d = constants['a'], constants['b'], constants['c'], constants['d']
    mu_te,  sig_te  = stats['thetae_mean'],     stats['thetae_std']
    mu_tes, sig_tes = stats['thetaestar_mean'], stats['thetaestar_std']
    mu_rh,  sig_rh  = stats['rh_mean'],         stats['rh_std']
    sig_tp,  mu_tp  = stats[f'{targetvar}_std'], stats[f'{targetvar}_mean']
    kappa      = -b * sig_te / sig_tes
    Theta_crit = mu_te + b * (sig_te / sig_tes) * mu_tes - c * sig_te
    alpha_hi   = sig_tp * a / sig_te**3
    rh_scale   = sig_te / sig_rh
    gamma_sfc  = d * sig_te
    latex = (
        r'\ell_{\mathrm{HI}} = \underbrace{' + f'{alpha_hi:.3g}' + r'}_{\alpha_{HI}}'
        r'\cdot\Bigl[\max\!\bigl('
        r'\underbrace{' + f'{rh_scale:.2f}' + r'}_{\sigma_{\theta_e}/\sigma_{RH}}'
        r'(\mathrm{RH} - \bar{\mathrm{RH}}),\;'
        r'\theta_e - \underbrace{' + f'{kappa:.3f}' + r'}_{\kappa}\theta_e^* - '
        r'\underbrace{' + f'{Theta_crit:.1f}' + r'\ \mathrm{K}}_{\Theta_e^{\mathrm{crit}}}'
        r'\bigr) + \underbrace{' + f'{gamma_sfc:.3f}' + r'}_{\gamma_{SFC}}'
        r'\cdot\max(\widetilde{LF},\,\widetilde{SHF})\Bigr]^3'
    )
    rows = [
        (r'$\alpha_{HI}$', f'{alpha_hi:.4g}', 'K⁻³ ℓ', 'Cubic precipitation sensitivity'),
        (r'$\kappa$', f'{kappa:.3f}', '—', 'θe* weighting; κ≈1 → CAPE-like onset criterion'),
        (r'$\Theta_e^{\mathrm{crit}}$', f'{Theta_crit:.1f}', 'K', 'Critical equivalent pot. temp. for onset'),
        (r'$\sigma_{\theta_e}/\sigma_{RH}$', f'{rh_scale:.3f}', 'K %⁻¹', 'Scale factor mapping RH to θe units'),
        (r'$\gamma_{SFC}$', f'{gamma_sfc:.4f}', 'K', 'Surface flux modifier in θe units; < 0 suppresses onset over land'),
    ]
    return latex, rows

PHYS_FNS = {
    'sr_bl':  _phys_sr_bl,
    'sr_lo':  _phys_sr_lo,
    'sr_med': _phys_sr_med,
    'sr_hi':  _phys_sr_hi,
}

In [9]:
from IPython.display import Math

if STATS is None or not registry:
    print('[warn] Need stats.json and optimized_equations.pkl — run on the cluster')
else:
    targetvar = CONFIGS['variables']['target']
    for name, eqspec in SRCONFIG['optimizedeqs'].items():
        opt = registry.get(name)
        fn  = PHYS_FNS.get(name)
        if opt is None or fn is None:
            continue
        latex_eq, rows = fn(opt['constants'], STATS, targetvar=targetvar)
        desc = eqspec['description']
        display(HTML(f'<b style="font-size:1.05em">{desc}</b>'))
        display(Math(r'P = \mathrm{expm1}\!\bigl(\max(0,\;\ell)\bigr),\quad' + latex_eq))
        df = pd.DataFrame(rows, columns=['Symbol', 'Value', 'Units', 'Description'])
        display(HTML(df.to_html(escape=False, index=False)))
        display(HTML('<br/>'))

<IPython.core.display.Math object>

Symbol,Value,Units,Description
$B_L^{\mathrm{crit}}$,-0.1141,m s⁻²,Critical buoyancy deficit for onset
$\alpha_{BL}$,1660,(m s⁻²)⁻³ ℓ,Cubic precipitation sensitivity
$\beta_{BL}$,-0.057,ℓ,Intercept in log-precipitation space


<IPython.core.display.Math object>

Symbol,Value,Units,Description
$\alpha_{LO}$,0.02381,ℓ,Exponential amplitude (absorbs centering)
$\sigma_{RH}$,21.96,%,RH scale — reciprocal is e-folding sensitivity
$\beta_{LO}$,-0.006,ℓ,Intercept in log-precipitation space


<IPython.core.display.Math object>

Symbol,Value,Units,Description
$\alpha_{MED}$,0.001328,K⁻³ ℓ,Cubic precipitation sensitivity
$\kappa$,0.794,—,θe* weighting; κ≈1 → CAPE-like onset criterion
$\Theta_e^{\mathrm{crit}}$,93.2,K,Critical equivalent pot. temp. for onset
$\sigma_{\theta_e}/\sigma_{RH}$,0.414,K %⁻¹,Scale factor mapping RH to θe units


<IPython.core.display.Math object>

Symbol,Value,Units,Description
$\alpha_{HI}$,0.0005555,K⁻³ ℓ,Cubic precipitation sensitivity
$\kappa$,-0.560,—,θe* weighting; κ≈1 → CAPE-like onset criterion
$\Theta_e^{\mathrm{crit}}$,569.3,K,Critical equivalent pot. temp. for onset
$\sigma_{\theta_e}/\sigma_{RH}$,0.414,K %⁻¹,Scale factor mapping RH to θe units
$\gamma_{SFC}$,2.3642,K,Surface flux modifier in θe units; < 0 suppresses onset over land


### Gaussian kernel parameters (nn_gauss)

Kernel center and width in sigma-level coordinates (σ ∈ [0.50, 1.00], surface = 1.00).
Conversion from learned parameters: σ_center = 0.75 + μ × 0.25, Δσ = exp(logstd) × 0.25.

In [10]:
import torch

nn_cfg    = CONFIGS['experiments']['nn']
fieldvars = nn_cfg['runs']['nn_gauss']['fieldvars']
modeldir  = os.path.join('..', CONFIGS['filepaths']['models'].split('monsoon-discovery/')[-1], 'nn')
seeds     = nn_cfg['seeds']

kernel_rows = []
for seed in seeds:
    ckpt_path = os.path.join(modeldir, f'nn_gauss_{seed}.pth')
    if not os.path.exists(ckpt_path):
        continue
    state = torch.load(ckpt_path, map_location='cpu')
    if 'state_dict' in state:
        state = state['state_dict']
    mu     = state['kernel.function.mu'].numpy()
    logstd = state['kernel.function.logstd'].numpy()
    for i, fv in enumerate(fieldvars):
        kernel_rows.append({
            'Seed':        seed,
            'Variable':    fv,
            'σ_center':    round(float(0.75 + mu[i]*0.25), 4),
            'Δσ (1σ width)': round(float(np.exp(logstd[i])*0.25), 4),
        })

if not kernel_rows:
    print('[warn] No nn_gauss checkpoints found — run on the cluster')
else:
    kdf = pd.DataFrame(kernel_rows)
    summary = kdf.groupby('Variable')[['σ_center','Δσ (1σ width)']].agg(['mean','std']).round(4)
    summary.columns = ['σ_center (mean)', 'σ_center (std)', 'Δσ mean', 'Δσ std']
    display(HTML(summary.to_html()))

,σ_center (mean),σ_center (std),Δσ mean,Δσ std
Variable,,,,
rh,-0.0410,0.1772,0.5374,0.0445
thetae,1.0198,0.0075,0.1459,0.0054
thetaestar,0.7707,0.0016,0.1279,0.0007
